# Wireless Communication Networks and Systems
## LAB 03: Digital Modulation (ASK, 2-FSK, BPSK)
*Python simulation*

---
### Learning Objectives
- Explain how bits are mapped to waveforms using ASK, 2-FSK, and BPSK.
- Generate modulated signals, visualize them in time and frequency.
- Build simple receivers (energy/envelope/correlation) to recover bits.
- Measure Bit Error Rate (BER) vs SNR on an AWGN channel and compare schemes.

### Background (Very Short)
Digital modulation maps bits onto a carrier by changing its amplitude (ASK), frequency (2-FSK), or phase (BPSK). In this lab we keep math light and emphasize intuition, plots, and code. We use an AWGN channel to study robustness.

### Global Parameters & Notation
- Bit rate $R_b$ (bits/s), bit duration $T_b = 1/R_b$.
- Oversampling factor $N_s$ (samples per bit), sampling freq. $f_s = N_s \cdot R_b$.
- Carrier frequency $f_c$ (ASK/BPSK). For 2-FSK, use $f_0, f_1$.

**Recommended defaults (safe for plots):**
$$R_b = 100\text{ b/s},\quad N_s = 100,\quad f_s = 10\text{ kHz},\quad f_c = 1\text{ kHz},\quad (f_0, f_1) = (800, 1200)\text{ Hz}$$


---
## Step 0 — Setup & Helper Functions
Paste and run this once at the top of your notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Global parameters (feel free to tweak) ----
Rb = 100          # bit rate [bits/s]
Ns = 100          # samples per bit
fs = Rb * Ns      # sampling frequency [Hz]
Tb = 1 / Rb       # bit duration [s]
fc = 1000         # carrier for ASK/BPSK [Hz]
f0, f1 = 800, 1200 # FSK mark/space [Hz]


def nrz_bits(bits):
    """Map bits {0,1} to NRZ levels {0,1} for ASK (OOK)."""
    return np.array(bits, dtype=float)


def nrz_bipolar(bits):
    """Map bits {0,1} to {-1,+1} for BPSK."""
    return 2 * np.array(bits, dtype=float) - 1.0


def upsample(symbols, Ns):
    """Repeat each symbol Ns times (NRZ rectangular pulse shaping)."""
    return np.repeat(symbols, Ns)


def time_axis(n_samples):
    return np.arange(n_samples) / fs


def carrier_wave(freq, n_samples, phase=0.0):
    t = time_axis(n_samples)
    return np.sin(2 * np.pi * freq * t + phase)


def rfft_mag(x):
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), d=1 / fs)
    return f, np.abs(X) / len(x)


def add_awgn(x, snr_db):
    """Add AWGN to reach target SNR (per-sample power basis)."""
    p_sig = np.mean(x**2)
    snr_lin = 10 ** (snr_db / 10)
    p_noise = p_sig / snr_lin
    noise = np.random.normal(0, np.sqrt(p_noise), size=len(x))
    return x + noise


def ber(ref_bits, rx_bits):
    ref = np.array(ref_bits).astype(int)
    rx = np.array(rx_bits).astype(int)
    return np.mean(ref != rx)


---
## Step 1 — Bitstream, Timebase & Sanity Plots

**Task:** Generate $N_b = 200$ random bits, build NRZ waveforms (rectangular pulses) and visualize them.


In [ ]:
Nb = 200
bits = np.random.randint(0, 2, size=Nb)

# NRZ for ASK (0/1) and BPSK (-1/+1)
nrz_ask = nrz_bits(bits)
nrz_bpsk = nrz_bipolar(bits)

# Upsample to waveform-level sequences
base_ask = upsample(nrz_ask, Ns)  # length = Nb*Ns
base_bpsk = upsample(nrz_bpsk, Ns)

t = time_axis(len(base_ask))

# Plot a small window (first 10 bits) for readability
win = 10 * Ns
plt.figure()
plt.plot(t[:win], base_ask[:win])
plt.title("NRZ baseband (ASK OOK levels)")
plt.xlabel("Time [s]")
plt.ylabel("Amp")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(t[:win], base_bpsk[:win])
plt.title("NRZ baseband (BPSK levels -1/+1)")
plt.xlabel("Time [s]")
plt.ylabel("Amp")
plt.grid(True)
plt.show()


### Record
Screenshots of the two baseband waveforms (first ~10 bits) with short captions.


**Observation / Answer:**


---
## Step 2 — ASK (OOK) Modulation & Envelope Detection

### Modulator
For On-Off Keying, multiply NRZ levels $\{0, 1\}$ by a carrier at $f_c$.


In [ ]:
c = carrier_wave(fc, len(base_ask))
s_ask = base_ask * c  # OOK

# Time view (short window)
plt.figure()
plt.plot(t[:win], s_ask[:win])
plt.title("ASK (OOK) waveform")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

# Spectrum (use a moderate slice for speed or whole sequence)
f, M = rfft_mag(s_ask)
plt.figure()
plt.plot(f, M)
plt.title("ASK spectrum (magnitude)")
plt.xlabel("Frequency [Hz]")
plt.ylabel("|X(f)|")
plt.grid(True)
plt.show()


### Channel + Receiver (envelope detection)
Rectify & integrate per bit, threshold at $\theta$.


In [ ]:
snr_db = 6
y_ask = add_awgn(s_ask, snr_db)

# Simple envelope: absolute value then integrate energy per bit
env = np.abs(y_ask)
rx_bits_ask = []
for i in range(Nb):
    seg = env[i * Ns : (i + 1) * Ns]
    metric = np.mean(seg)  # average envelope in bit window
    rx_bits_ask.append(1 if metric > 0.3 else 0)  # threshold tweakable

print("ASK (OOK) BER @", snr_db, "dB =", ber(bits, rx_bits_ask))


### Expected outputs
ASK time plot, spectrum, chosen threshold, BER at 0, 4, 8, 12 dB.


In [ ]:
# Evaluate ASK BER at 0, 4, 8, 12 dB
for snr in [0, 4, 8, 12]:
    y_test = add_awgn(s_ask, snr)
    env_test = np.abs(y_test)
    rx_test = [1 if np.mean(env_test[i * Ns : (i + 1) * Ns]) > 0.3 else 0 for i in range(Nb)]
    print(f"ASK (OOK) BER @ {snr:2d} dB = {ber(bits, rx_test):.4f}")


**Observation / Answer:**


---
## Step 3 — 2-FSK Modulation & Noncoherent Energy Detection

### Modulator
Use $f_1$ for bit 1, $f_0$ for bit 0 (same amplitude).


In [ ]:
# Build per-bit FSK by stitching bit-length carriers
s_fsk = np.zeros(Nb * Ns)
for i, b in enumerate(bits):
    fbit = f1 if b == 1 else f0
    s_fsk[i * Ns : (i + 1) * Ns] = carrier_wave(fbit, Ns)

plt.figure()
plt.plot(t[:win], s_fsk[:win])
plt.title("2-FSK waveform")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

f, M = rfft_mag(s_fsk)
plt.figure()
plt.plot(f, M)
plt.title("2-FSK spectrum")
plt.xlabel("Frequency [Hz]")
plt.ylabel("|X(f)|")
plt.grid(True)
plt.show()


### Channel + Receiver (energy in narrowband filters)
Compare energy around $f_0$ vs $f_1$.


In [ ]:
def tone_energy(x, f_tone):
    # Correlate with in-phase and quadrature sinusoids, sum energies in a bit window
    tloc = np.arange(Ns) / fs
    cI = np.sin(2 * np.pi * f_tone * tloc)
    cQ = np.cos(2 * np.pi * f_tone * tloc)
    E = 0.0
    for i in range(Nb):
        seg = x[i * Ns : (i + 1) * Ns]
        I = np.dot(seg, cI)
        Q = np.dot(seg, cQ)
        E += I**2 + Q**2  # optional: keep per-bit
    return E


snr_db = 6
y_fsk = add_awgn(s_fsk, snr_db)

rx_bits_fsk = []
tloc = np.arange(Ns) / fs
cI0, cQ0 = np.sin(2 * np.pi * f0 * tloc), np.cos(2 * np.pi * f0 * tloc)
cI1, cQ1 = np.sin(2 * np.pi * f1 * tloc), np.cos(2 * np.pi * f1 * tloc)

for i in range(Nb):
    seg = y_fsk[i * Ns : (i + 1) * Ns]
    I0, Q0 = np.dot(seg, cI0), np.dot(seg, cQ0)
    I1, Q1 = np.dot(seg, cI1), np.dot(seg, cQ1)
    E0, E1 = I0**2 + Q0**2, I1**2 + Q1**2
    rx_bits_fsk.append(1 if E1 > E0 else 0)

print("2-FSK BER @", snr_db, "dB =", ber(bits, rx_bits_fsk))


### Expected outputs
FSK time plot, spectrum, BER at 0, 4, 8, 12 dB.


In [ ]:
# Evaluate 2-FSK BER at 0, 4, 8, 12 dB
for snr in [0, 4, 8, 12]:
    y_test = add_awgn(s_fsk, snr)
    rx_test = []
    for i in range(Nb):
        seg = y_test[i * Ns : (i + 1) * Ns]
        I0, Q0 = np.dot(seg, cI0), np.dot(seg, cQ0)
        I1, Q1 = np.dot(seg, cI1), np.dot(seg, cQ1)
        rx_test.append(1 if (I1**2 + Q1**2) > (I0**2 + Q0**2) else 0)
    print(f"2-FSK BER @ {snr:2d} dB = {ber(bits, rx_test):.4f}")


**Observation / Answer:**


---
## Step 4 — BPSK Modulation & Coherent Correlator Receiver

### Modulator
Map bits to $\{-1, +1\}$, then multiply by carrier at $f_c$.


In [ ]:
s_bpsk = upsample(nrz_bipolar(bits), Ns) * carrier_wave(fc, Nb * Ns)

plt.figure()
plt.plot(t[:win], s_bpsk[:win])
plt.title("BPSK waveform")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

f, M = rfft_mag(s_bpsk)
plt.figure()
plt.plot(f, M)
plt.title("BPSK spectrum")
plt.xlabel("Frequency [Hz]")
plt.ylabel("|X(f)|")
plt.grid(True)
plt.show()


### Channel + Receiver (coherent correlation)
Correlate per bit with known carrier.


In [ ]:
snr_db = 6
y_bpsk = add_awgn(s_bpsk, snr_db)

rx_bits_bpsk = []
cbit = carrier_wave(fc, Ns)  # reference carrier for one-bit window

for i in range(Nb):
    seg = y_bpsk[i * Ns : (i + 1) * Ns]
    metric = np.dot(seg, cbit)  # integrate over bit
    rx_bits_bpsk.append(1 if metric >= 0 else 0)

print("BPSK BER @", snr_db, "dB =", ber(bits, rx_bits_bpsk))


### Expected outputs
BPSK time plot, spectrum, BER at 0, 4, 8, 12 dB.


In [ ]:
# Evaluate BPSK BER at 0, 4, 8, 12 dB
for snr in [0, 4, 8, 12]:
    y_test = add_awgn(s_bpsk, snr)
    rx_test = [1 if np.dot(y_test[i * Ns : (i + 1) * Ns], cbit) >= 0 else 0 for i in range(Nb)]
    print(f"BPSK BER @ {snr:2d} dB = {ber(bits, rx_test):.4f}")


**Observation / Answer:**


---
## Step 5 — BER vs SNR (Mini Experiment)

**Task:** Sweep SNR and compare schemes on the same bit sequence.


In [ ]:
SNRs = [0, 2, 4, 6, 8, 10, 12]
ber_ask, ber_fsk, ber_bpsk = [], [], []

# Precompute fixed waveforms (before noise)
s_ask = upsample(nrz_bits(bits), Ns) * carrier_wave(fc, Nb * Ns)
s_fsk = np.concatenate([carrier_wave(f1 if b else f0, Ns) for b in bits])
s_bpsk = upsample(nrz_bipolar(bits), Ns) * carrier_wave(fc, Nb * Ns)
cbit = carrier_wave(fc, Ns)
tloc = np.arange(Ns) / fs
cI0, cQ0 = np.sin(2 * np.pi * f0 * tloc), np.cos(2 * np.pi * f0 * tloc)
cI1, cQ1 = np.sin(2 * np.pi * f1 * tloc), np.cos(2 * np.pi * f1 * tloc)

for snr_db in SNRs:
    # ASK receive
    y = add_awgn(s_ask, snr_db)
    env = np.abs(y)
    rxA = []
    for i in range(Nb):
        m = np.mean(env[i * Ns : (i + 1) * Ns])
        rxA.append(1 if m > 0.3 else 0)
    ber_ask.append(ber(bits, rxA))

    # FSK receive
    y = add_awgn(s_fsk, snr_db)
    rxF = []
    for i in range(Nb):
        seg = y[i * Ns : (i + 1) * Ns]
        I0, Q0 = np.dot(seg, cI0), np.dot(seg, cQ0)
        I1, Q1 = np.dot(seg, cI1), np.dot(seg, cQ1)
        rxF.append(1 if (I1**2 + Q1**2) > (I0**2 + Q0**2) else 0)
    ber_fsk.append(ber(bits, rxF))

    # BPSK receive
    y = add_awgn(s_bpsk, snr_db)
    rxP = []
    for i in range(Nb):
        seg = y[i * Ns : (i + 1) * Ns]
        rxP.append(1 if np.dot(seg, cbit) >= 0 else 0)
    ber_bpsk.append(ber(bits, rxP))

print("SNRs:", SNRs)
print("BER ASK :", ber_ask)
print("BER 2FSK:", ber_fsk)
print("BER BPSK:", ber_bpsk)

# Optional: plot BER curves (linear y for simplicity)
plt.figure()
plt.plot(SNRs, ber_ask, marker='o', label='ASK (OOK)')
plt.plot(SNRs, ber_fsk, marker='o', label='2-FSK')
plt.plot(SNRs, ber_bpsk, marker='o', label='BPSK')
plt.xlabel("SNR [dB]")
plt.ylabel("BER")
plt.title("BER vs SNR (AWGN)")
plt.legend()
plt.grid(True)
plt.show()


### Record
The printed BER lists and the BER plot. Explain which scheme is most robust and why.


**Observation / Answer:**


---
## Step 6 — Optional Explorations (Pick One)

- **Option 1: Constellation view (BPSK)**. Scatter of correlator outputs ($I$) across noise levels.
- **Option 2: Pulse shaping**. Replace rectangular NRZ with raised-cosine (conceptual) and discuss bandwidth.
- **Option 3: Carrier offset**. Add a small frequency offset to BPSK carrier at the receiver and observe BER.


In [ ]:
# Option 3: Compare BPSK BER vs SNR with carrier frequency offset
SNRs = [0, 2, 4, 6, 8, 10, 12]
offsets = [0, 20, 50]

plt.figure(figsize=(7, 4))
for df in offsets:
    cbit_offset = carrier_wave(fc + df, Ns)
    ber_curve = []
    for snr in SNRs:
        y = add_awgn(s_bpsk, snr)
        rx = [1 if np.dot(y[i * Ns : (i + 1) * Ns], cbit_offset) >= 0 else 0 for i in range(Nb)]
        ber_curve.append(ber(bits, rx))
    plt.plot(SNRs, ber_curve, marker='o', label=f'Offset = {df} Hz')

plt.xlabel('SNR [dB]')
plt.ylabel('BER')
plt.title('BPSK BER vs SNR with Carrier Frequency Offset')
plt.legend()
plt.grid(True)
plt.show()


**Observation / Answer:**
With no or small frequency offset (0 Hz, 20 Hz), BPSK performs normally and the BER drops to 0 as SNR increases. However, with a larger offset (50 Hz), the receiver carrier drifts out of phase with the signal, causing the detector to fail and keeping the BER stuck near 0.5 across all SNR levels.


---
## Guiding Questions


### Question 1
Why does BPSK typically outperform ASK at the same SNR?


**Answer:**
BPSK uses antipodal signaling (+1 and -1), providing twice the distance between symbol states compared to ASK (0 and 1) for the same signal power. This larger separation makes BPSK significantly more resilient to noise.


### Question 2
When might 2-FSK be preferred over BPSK (hint: noncoherent detection, narrowband interference)?


**Answer:**
2-FSK is preferred when simple noncoherent detection is needed without complex carrier phase recovery, or in channels with strong narrowband interference and amplitude fading.


### Question 3
How do carrier frequency choices ($f_c, f_0, f_1$) and bit rate ($R_b$) affect spectral occupancy?


**Answer:**
The bit rate Rb sets the bandwidth of the main spectral lobe around the carrier. The carrier frequencies (fc or f0, f1) determine where the spectrum is centered, with 2-FSK requiring wider total bandwidth to separate the two frequency tones.


### Question 4
What practical issues would appear in a real radio (timing, phase/frequency offsets, filtering)?


**Answer:**
Real radios face carrier frequency and phase offsets, symbol timing errors (sampling jitter), and bandlimiting filter imperfections, which cause phase drift and inter-symbol interference (ISI).
